# Solution 02 — basic consumer

In [ ]:
from confluent_kafka import Consumer

consumer = Consumer({
    'bootstrap.servers': 'redpanda:29092',
    'group.id':          'energy-monitor',
    'auto.offset.reset': 'earliest',
})
consumer.subscribe(['strom', 'wasser'])
print('Subscribed.')

## Step 3 — read existing messages

In [ ]:
messages_read, empty_polls = 0, 0
while messages_read < 20 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1
    key   = msg.key().decode()   if msg.key()   else 'None'
    value = msg.value().decode() if msg.value() else 'None'
    print(f'[{messages_read}] {msg.topic()} P{msg.partition()} '
          f'offset={msg.offset()} key={key}')
    print(f'     {value}')

## Task B — fresh group reads from the start

Notice this consumer reads the *same* messages again, despite the first one having read them already. The new `group.id` has no stored offset, so `auto.offset.reset='earliest'` kicks in.

In [ ]:
consumer.close()
consumer2 = Consumer({
    'bootstrap.servers': 'redpanda:29092',
    'group.id':          'analytics',
    'auto.offset.reset': 'earliest',
})
consumer2.subscribe(['strom', 'wasser'])
for i in range(10):
    msg = consumer2.poll(2.0)
    if msg and not msg.error():
        print(f'[{i+1}] {msg.topic()} P{msg.partition()} '
              f'offset={msg.offset()}')
consumer2.close()